# Load statistical baselines (Darts)



In [ ]:
!pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"load"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

from darts import TimeSeries, concatenate
from darts.models import NaiveSeasonal, AutoARIMA, AutoTheta, AutoETS
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from preprocessing import load_data
from gefcom import load_gefcom_load, evaluate_gefcom_load
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
NUM_SAMPLES=100

OUTPUT_DIR=ROOT/"outputs"/"predictions"
MODEL_DIR=ROOT/"models"/"darts"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL_DIR.mkdir(parents=True,exist_ok=True)

def build_stat(name):
    if name=="auto_arima":
        return AutoARIMA(
            seasonal=True,
            season_length=24,
            stepwise=True,
            trace=True,
            d=1,D=1,
            max_p=1,max_q=1,max_P=1,max_Q=1,
        )
    if name=="auto_theta":
        return AutoTheta(season_length=24)
    if name=="auto_ets":
        return AutoETS(season_length=24)
    raise ValueError(name)

STAT_CLASSES={"auto_arima":AutoARIMA,"auto_theta":AutoTheta,"auto_ets":AutoETS}

def probabilistic_frame(series,forecast):
    gt=series.slice_intersect(forecast)
    fc=forecast.slice_intersect(gt)
    out=pd.DataFrame({
        "item_id":1,
        "timestamp":gt.time_index,
        "target":gt.values().flatten(),
    })
    for q in OUTPUT_QUANTILES:
        out[str(q)]=fc.quantile(q).values().flatten()
    out["prediction"]=out["0.5"]
    return out


## GEFCom2014-L


In [ ]:
gef_df=load_gefcom_load(ROOT/"data"/"gefcom_load_data.csv")
gef_series=TimeSeries.from_dataframe(
    gef_df,time_col="timestamp",value_cols="target",freq="h"
).astype(np.float32)

# Each competition task is refit on all realized load history available before it.
def gef_history_before(task_start):
    return gef_series.drop_after(pd.Timestamp(task_start))

def make_gef_frame(task_id,task_df,forecast,probabilistic):
    if probabilistic:
        d=pd.DataFrame({
            "item_id":1,
            "timestamp":forecast.time_index,
            "task_id":task_id,
        })
        for q in BASE_QUANTILES:
            d[str(q)]=forecast.quantile(q).values().flatten()
        d["prediction"]=d["0.5"]
    else:
        d=pd.DataFrame({
            "item_id":1,
            "timestamp":forecast.time_index,
            "task_id":task_id,
            "prediction":forecast.values().flatten(),
        })
    return d


In [ ]:
# Seasonal Naive
frames=[]
for task_id in range(2,17):
    task_df=gef_df[gef_df["task_id"]==task_id].sort_values("timestamp")
    task_start=task_df["timestamp"].min()
    history=gef_history_before(task_start)
    if history.end_time()!=pd.Timestamp(task_start)-pd.Timedelta(hours=1):
        raise ValueError(f"Bad GEFCom-L history alignment for task {task_id}.")
    model=NaiveSeasonal(K=24)
    model.fit(history)
    fc=model.predict(len(task_df))
    frames.append(make_gef_frame(task_id,task_df,fc,False))

gef_naive=pd.concat(frames,ignore_index=True).merge(
    gef_df[["item_id","timestamp","task_id","target"]],
    on=["item_id","timestamp","task_id"],
    how="inner",
)
gef_naive.to_csv(OUTPUT_DIR/"seasonal_naive_gefcom_load.csv",index=False)
print("GEFCom-L Seasonal Naive")
print("MAE:",mean_absolute_error(gef_naive.target,gef_naive.prediction))
print("RMSE:",root_mean_squared_error(gef_naive.target,gef_naive.prediction))
print("aQL: N/A (deterministic baseline)")


In [ ]:
for name in ["auto_arima","auto_theta","auto_ets"]:
    frames=[]
    for task_id in range(2,17):
        task_df=gef_df[gef_df["task_id"]==task_id].sort_values("timestamp")
        task_start=task_df["timestamp"].min()
        history=gef_history_before(task_start)

        model=build_stat(name)
        model.fit(history)
        fc=model.predict(
            len(task_df),
            num_samples=NUM_SAMPLES,
        )
        frames.append(make_gef_frame(task_id,task_df,fc,True))

    result=pd.concat(frames,ignore_index=True).merge(
        gef_df[["item_id","timestamp","task_id","target"]],
        on=["item_id","timestamp","task_id"],
        how="inner",
    )

    expected=sum(len(gef_df[gef_df["task_id"]==i]) for i in range(2,17))
    if len(result)!=expected:
        raise ValueError(f"Expected {expected} GEFCom-L rows, got {len(result)}.")

    result.to_csv(OUTPUT_DIR/f"{name}_gefcom_load.csv",index=False)
    summary=evaluate_gefcom_load(result,BASE_QUANTILES)
    display(summary[["task_id","raw_aql","normalized_aql"]])
    summary.to_csv(OUTPUT_DIR/f"{name}_gefcom_load_summary.csv",index=False)


## PG&E


In [ ]:
LOAD_EXISTING=False
SAVE_MODELS=True
PGE_MODEL_DIR=MODEL_DIR/"pge"
PGE_MODEL_DIR.mkdir(parents=True,exist_ok=True)

data=load_data(ROOT/"data")
series=TimeSeries.from_dataframe(
    data.full,time_col="timestamp",value_cols="target",freq="h"
).astype(np.float32)
future_covariates=TimeSeries.from_dataframe(
    data.full,time_col="timestamp",value_cols=data.covariates,freq="h"
).astype(np.float32)

test_start=pd.Timestamp(data.test["timestamp"].min())
stat_train,_=series.split_after(test_start-series.freq)
future_train,_=future_covariates.split_after(test_start-series.freq)

print("Target:",series.start_time(),"->",series.end_time())
print("Future covariates:",future_covariates.start_time(),"->",future_covariates.end_time())
assert future_covariates.end_time()>=series.end_time()


In [ ]:
seasonal_naive=NaiveSeasonal(K=24)
seasonal_naive.fit(stat_train)
bt=seasonal_naive.historical_forecasts(
    series=series,
    start=test_start,
    forecast_horizon=24,
    stride=24,
    retrain=True,
    last_points_only=False,
    verbose=True,
)
forecast=concatenate(bt)
gt=series.slice_intersect(forecast)
fc=forecast.slice_intersect(gt)
result=pd.DataFrame({
    "item_id":1,
    "timestamp":gt.time_index,
    "target":gt.values().flatten(),
    "prediction":fc.values().flatten(),
})
for q in OUTPUT_QUANTILES:
    result[str(q)]=np.nan
result.to_csv(OUTPUT_DIR/"seasonal_naive.csv",index=False)
print("Seasonal Naive")
print(f"MAE:  {mean_absolute_error(result.target,result.prediction):.6f}")
print(f"RMSE: {root_mean_squared_error(result.target,result.prediction):.6f}")
print("aQL:  N/A (deterministic baseline)")


In [ ]:
for name in ["auto_arima","auto_theta","auto_ets"]:
    path=PGE_MODEL_DIR/f"{name}.darts"

    if LOAD_EXISTING and path.exists():
        model=STAT_CLASSES[name].load(str(path))
    else:
        model=build_stat(name)
        model.fit(
            series=stat_train,
            future_covariates=future_train,
        )
        if SAVE_MODELS:
            model.save(str(path))

    bt=model.historical_forecasts(
        series=series,
        future_covariates=future_covariates,
        start=test_start,
        forecast_horizon=24,
        stride=24,
        num_samples=NUM_SAMPLES,
        retrain=False,
        last_points_only=False,
        verbose=True,
    )
    forecast=concatenate(bt)
    result=probabilistic_frame(series,forecast)
    result.to_csv(OUTPUT_DIR/f"{name}.csv",index=False)
    print_metrics(result,BASE_QUANTILES,label=name)
